In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
import os

# 1. Define Data Transformations
# Resize images to 224x224 (expected by ResNet) and normalize using ImageNet standards.
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(), # Data augmentation
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

# 2. Setup DataLoaders
# Using os.getcwd() because __file__ does not exist in Jupyter/Interactive environments
base_dir = os.getcwd()

# Adjusting to the nested Kaggle extraction structure: chest_xray/chest_xray
data_dir = os.path.join(base_dir, 'chest_xray', 'chest_xray')

print(f"Looking for data in: {data_dir}") 

image_datasets = {
    x: datasets.ImageFolder(os.path.join(data_dir, x), data_transforms[x])
    for x in ['train', 'val']
}

dataloaders = {
    x: DataLoader(image_datasets[x], batch_size=32, shuffle=True, num_workers=2)
    for x in ['train', 'val']
}

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")

# 3. Initialize Pre-trained ResNet18
# Using weights parameter instead of the deprecated pretrained=True
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Freeze the feature extraction layers so we only train the final classifier
for param in model.parameters():
    param.requires_grad = False

# Replace the final fully connected layer (binary classification: Normal vs Pneumonia)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 2)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

# 4. Training Loop
num_epochs = 20 # Keep it low for initial testing; increase to 10-20 for better accuracy

for epoch in range(num_epochs):
    print(f'\nEpoch {epoch+1}/{num_epochs}')
    print('-' * 10)

    for phase in ['train', 'val']:
        if phase == 'train':
            model.train()
        else:
            model.eval()

        running_loss = 0.0
        running_corrects = 0

        for inputs, labels in dataloaders[phase]:
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()

            # Forward pass
            with torch.set_grad_enabled(phase == 'train'):
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                loss = criterion(outputs, labels)

                # Backward pass + optimize only if in training phase
                if phase == 'train':
                    loss.backward()
                    optimizer.step()

            # Statistics
            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)

        epoch_loss = running_loss / len(image_datasets[phase])
        epoch_acc = running_corrects.double() / len(image_datasets[phase])
        
        print(f'{phase.capitalize()} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

# 5. Export the Weights for the API
torch.save(model.state_dict(), 'model_weights.pth')
print("\nSuccess! Model weights saved to model_weights.pth")

Looking for data in: C:\Users\ahada\Downloads\ML_practice\projects\Production_Ready_Medical_API\chest_xray\chest_xray
Training on device: cuda:0

Epoch 1/20
----------
Train Loss: 0.2331 Acc: 0.9112
Val Loss: 0.4876 Acc: 0.6875

Epoch 2/20
----------
Train Loss: 0.1604 Acc: 0.9383
Val Loss: 0.4736 Acc: 0.6875

Epoch 3/20
----------
Train Loss: 0.1467 Acc: 0.9433
Val Loss: 0.3493 Acc: 0.7500

Epoch 4/20
----------
Train Loss: 0.1353 Acc: 0.9475
Val Loss: 0.2985 Acc: 0.7500

Epoch 5/20
----------
Train Loss: 0.1232 Acc: 0.9515
Val Loss: 0.3525 Acc: 0.8125

Epoch 6/20
----------
Train Loss: 0.1288 Acc: 0.9511
Val Loss: 0.3103 Acc: 0.7500

Epoch 7/20
----------
Train Loss: 0.1145 Acc: 0.9546
Val Loss: 0.5007 Acc: 0.8125

Epoch 8/20
----------
Train Loss: 0.1236 Acc: 0.9511
Val Loss: 0.3680 Acc: 0.8125

Epoch 9/20
----------
Train Loss: 0.1203 Acc: 0.9534
Val Loss: 0.2566 Acc: 0.8125

Epoch 10/20
----------
Train Loss: 0.1145 Acc: 0.9559
Val Loss: 0.4637 Acc: 0.8125

Epoch 11/20
----------
